# Mamba-1.4B RLF — Train & Export for Mobile Edge Deployment

This notebook walks through the full pipeline:
1. Environment setup
2. Train / fine-tune a Mamba-1.4B model with **Recursive Latent Forcing (RLF)**
3. Export weights to the `.mamba.bin` format consumed by the mobile C engine
4. Export the BPE tokenizer to `.bpe.bin`
5. Verify the exported artifacts

> **Audience:** ML engineers on any compute backend:
> - CUDA GPU (Colab T4/A100, RTX 3090/4090, …) — needed for training
> - Apple Silicon M-series Macs (MPS backend) — inference/export only
> - CPU-only hosts — export/verification only
>
> **⚠ Training requires CUDA.** `mamba-ssm` has hard CUDA dependencies and
> cannot be installed on macOS or CPU-only hosts. Run training cells on a
> CUDA Linux machine (Colab, Lambda Labs, etc.) and use this notebook on
> your Mac for export and verification only.
>
> **Prereqs:** A trained base checkpoint or willingness to train from scratch.
> The 3-phase RLF trainer is at `rlf_trainer_1_4b.py` in this repo.


## 1. Environment Setup

> **Colab / CUDA users (TRAINING):**
> ```bash
> !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
> !pip install "transformers safetensors datasets ninja sentencepiece tiktoken accelerate einops"
> # mamba-ssm has NO Python 3.13 wheels (sdist missing csrc/ → 404). Do NOT pip install it.
> # Use install_dependencies() below (recursive source build) or see Cell 1b.
> !pip install transformers safetensors datasets ninja sentencepiece tiktoken
> ```
>
> **macOS / CPU users (EXPORT ONLY — cannot train):**
> `install_dependencies()` (Cell A) will install CPU-only packages.
> Training cells will be auto-skipped. Use this notebook on a CUDA machine
> for training, then run export/verification cells here.

> **Colab T4 bootstrap (run first):**
> 1. `Runtime → Change runtime type → T4 GPU`.
> 2. `!git clone <your-fork-url> && %cd Mamba2-Recursive-Latent-Forcing-Operates` (so `REPO_ROOT` resolves to repo root).
> 3. Optional persistence: `from google.colab import drive; drive.mount('/content/drive')` and point `RLF_CKPT_DIR` at Drive (see Cell 2a).
> 4. `!nvidia-smi` to confirm T4 (`sm_75` → notebook picks `fp16`; engine `bf16` defaults are overridden for smoke — see Cell 2b/2d).


### 1b. Colab T4: build `causal-conv1d` + `mamba-ssm` from source (Python 3.13 has no wheels)

Run the `install_dependencies()` cell below — on CUDA it automatically falls back to a recursive source build:
`/tmp/causal-conv1d` (v1.5.0) then `/tmp/mamba` (v2.2.4) with `TORCH_CUDA_ARCH_LIST=7.5 MAX_JOBS=2` (~5–15 min on T4).
Keep stock Colab torch (do NOT downgrade to cu121) — the build targets your installed torch.


In [ ]:
import subprocess, sys, json, os, textwrap
from pathlib import Path

# Colab: CWD is /content after fresh VM. Resolve to repo root containing rlf_engine_1_4b.py.
REPO_ROOT = Path(".").resolve()
if not (REPO_ROOT / "rlf_engine_1_4b.py").exists():
    # Try common Colab clone locations
    for cand in [Path("/content/rlf"), Path("/content/Mamba2-Recursive-Latent-Forcing-Operates"), Path("/content/mamba2backbonerecursion")]:
        if (cand / "rlf_engine_1_4b.py").exists():
            REPO_ROOT = cand
            break
print(f"REPO_ROOT={REPO_ROOT}")
# Optional: HF cache persistence on Drive to avoid re-downloads (set before any HF call):
# os.environ.setdefault("HF_HOME", "/content/drive/MyDrive/.hf_cache")
# NOTE: export_mamba_1_4b.py / notebooks/ / mobile/ are UNTRACKED in git origin.
# If REPO_ROOT lacks them after git clone, upload via Colab Files or `git pull` after pushing from your Mac.


Install runtime dependencies. Adjust versions to match your CUDA / PyTorch setup.


In [ ]:
def install_dependencies():
    """Install deps with platform-aware fallbacks (Colab T4 first-class)."""
    import platform, subprocess, sys

    arch   = platform.machine()
    system = platform.system()
    cuda_ok = False
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"],
            stderr=subprocess.STDOUT, text=True, timeout=10,
        )
        cuda_ok = bool(out.strip())
    except Exception:
        pass

    # requirements.txt parity: accelerate/einops/causal-conv1d are required by mamba_ssm.
    base_pkgs = [
        "transformers>=4.30.0",
        "safetensors",
        "datasets",
        "accelerate>=0.27.0",
        "einops>=0.7.0",
        "ninja",
        "sentencepiece",
        "tiktoken",
    ]

    if cuda_ok:
        # Keep Colab stock torch (cu128) — do NOT force cu121 downgrade (breaks preinstalled deps).
        print("[install] CUDA detected — installing pure-PyPI deps first (no torch downgrade)")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        ] + base_pkgs)
        # mamba-ssm has NO cp313 wheels: sdist is missing csrc/selective_scan.cpp → HTTP 404.
        # Try fast path (sdist) briefly, then fall back to recursive source build.
        import os as _os
        env = dict(_os.environ, TORCH_CUDA_ARCH_LIST="7.5", MAX_JOBS="2")
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                "causal-conv1d>=1.1.0", "mamba-ssm>=2.2.0",
            ], timeout=180)
            print("[install] mamba-ssm sdist worked (unexpected on py3.13) — done.")
        except Exception as e:
            print(f"[install] sdist failed as expected on py3.13 ({e}).")
            print("[install] Recursive source build: /tmp/causal-conv1d v1.5.0 → /tmp/mamba v2.2.4 (~5-15 min on T4)")
            subprocess.check_call(["rm", "-rf", "/tmp/mamba", "/tmp/causal-conv1d"])
            subprocess.check_call(["git", "clone", "--recursive", "--depth", "1", "--branch", "v1.5.0",
                                   "https://github.com/Dao-AILab/causal-conv1d.git", "/tmp/causal-conv1d"])
            subprocess.check_call(["git", "clone", "--recursive", "--depth", "1", "--branch", "v2.2.4",
                                   "https://github.com/state-spaces/mamba.git", "/tmp/mamba"])
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
                                   "--no-build-isolation", "/tmp/causal-conv1d"], env=env)
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
                                   "--no-build-isolation", "/tmp/mamba"], env=env)
            print("[install] Source build complete. Verify: python -c 'import mamba_ssm; print(mamba_ssm.__version__)'")
    else:
        base_pkgs.insert(0, "torch>=2.0.0")
        print("[install] No CUDA — installing CPU-only stack")
        print("[WARN] mamba-ssm requires CUDA and cannot be installed on this platform.")
        print("[WARN] Training cells will be skipped. Use export/verification cells only.")
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
            ] + base_pkgs)
        except subprocess.CalledProcessError as exc:
            print(f"[WARN] pip install failed: {exc}")
        print("[install] CPU-only packages installed. mamba-ssm skipped.")

install_dependencies()


Verify GPU availability and report VRAM.


In [ ]:
def detect_environment():
    """Detect available compute backend and return (DEVICE, DTYPE)."""
    import torch

    if torch.cuda.is_available():
        DEVICE = "cuda"
        torch.cuda.set_per_process_memory_fraction(0.85)  # T4 16GB: leave headroom for context/frag
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        props = torch.cuda.get_device_properties(0)
        print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
        cc = props.major * 10 + props.minor
        DTYPE = torch.bfloat16 if cc >= 80 else torch.float16  # T4 is sm_75 -> fp16; engine bf16 defaults overridden at model build for smoke
        print(f"Compute capability: sm_{props.major}{props.minor} → dtype={DTYPE}")
    elif hasattr(torch, "mps") and torch.mps.is_available():
        DEVICE = "mps"
        DTYPE  = torch.float32
        print("Using Apple Silicon MPS backend")
        free, _ = torch.mps.get_mem_info()
        print(f"MPS free memory: {free / 1e9:.0f} GB")
    else:
        DEVICE = "cpu"
        DTYPE  = torch.float32
        print("⚠  Using CPU backend — training will be very slow or impossible.")
        print("   mamba-ssm requires CUDA. Use a Colab T4/A100 instance for training.")

    print(f"DEVICE={DEVICE}  DTYPE={DTYPE}")
    return DEVICE, DTYPE


DEVICE, DTYPE = detect_environment()


In [ ]:
def safe_import_ml_stack():
    """Import ML stack with actionable error messages."""
    try:
        import torch
        import mamba_ssm
        from mamba_ssm import MambaLMHeadModel
        print("torch and mamba-ssm imported successfully.")
        return torch, mamba_ssm
    except ImportError as e:
        print(f"Import error: {e}")
        if "mamba_ssm" in str(e) or "mamba" in str(e).lower():
            print()
            print("mamba-ssm requires CUDA and cannot run on this platform.")
            print("Training cells will be skipped. You can still run export/verification.")
            print()
            print("To train, use a CUDA-enabled Linux machine (e.g., Colab):")
            print("  https://colab.research.google.com")
        raise ImportError("ML stack incomplete — see messages above.")

try:
    torch, mamba_ssm = safe_import_ml_stack()
except ImportError:
    print("[NOTE] Continuing without mamba-ssm. Training cells will need to be skipped.")
    torch = None
    mamba_ssm = None


## 2. Clone / Use the Existing RLF Training Code

The repo already ships a 3-phase RLF trainer for Mamba-1.4B:

| Phase | Purpose | Steps | Trainable params |
|-------|---------|-------|-----------------|
| 3a | Scratchpad warmup | 2,000 | latent_memory + bridge (~38K) |
| 3b | RLF joint training | 8,000 | top LoRA + loop engine + memory + bridge |
| 3c | SFT recovery | 1,000 | lm_head only |

**Total:** ~11,000 steps, ~7 hours on an RTX 3080 10 GB.

You can either:
- Run the trainer directly: `python rlf_trainer_1_4b.py --phase all`
- Or step through the phases from this notebook using the cells below.


### 2a. Configuration


In [ ]:
SFT_CKPT_DIR = REPO_ROOT / "checkpoints" / "sft_base"
RLF_CKPT_DIR = REPO_ROOT / "checkpoints" / "rlf_1_4b"
RLF_CKPT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL   = "state-spaces/mamba-1.4b"
# DEVICE and DTYPE are set by detect_environment() earlier in this notebook
D_MODEL      = 2048
N_LAYERS     = 48
BASE_SPLIT   = 24        # freeze bottom half
LORA_RANK    = 8
PREFIX_M     = 8
BRIDGE_RANK  = 64
MAX_LOOPS    = 6

LOOP_D_STATE = 16
LOOP_D_CONV  = 4
LOOP_EXPAND  = 1

# ── Training phases ───────────────────────────────────────────────────────────
# Full training: 11K steps over 3 phases (~7 hours on RTX 3080)
PHASE_CONFIG = {
    "3a": {"steps": 2000,  "lr_mem": 1e-3,  "lr_bridge": 5e-4, "lr_other": 0.0},
    "3b": {"steps": 8000,  "lr_mem": 5e-4,  "lr_bridge": 2e-4, "lr_other": 1e-4},
    "3c": {"steps": 1000,  "lr_mem": 0.0,   "lr_bridge": 0.0,  "lr_other": 1e-5},
}

# ── Smoke-test overrides (set to False for real runs) ─────────────────────────
SMOKE_TEST      = True     # toggle off for real training
SMOKE_TEST_SIZE = 100      # tiny dataset for quick verification
SMOKE_PHASE_CONFIG = {
    "3a": {"steps": 50,   "lr_mem": 1e-3,  "lr_bridge": 5e-4, "lr_other": 0.0},
    "3b": {"steps": 100,  "lr_mem": 5e-4,  "lr_bridge": 2e-4, "lr_other": 1e-4},
    "3c": {"steps": 25,   "lr_mem": 0.0,   "lr_bridge": 0.0,  "lr_other": 1e-5},
}
ACTIVE_PHASE_CONFIG = SMOKE_PHASE_CONFIG if SMOKE_TEST else PHASE_CONFIG

BATCH_SIZE   = 1
GRAD_ACCUM   = 8
LOG_EVERY    = 25
CKPT_EVERY   = 500

# ── Colab persistence ─────────────────────────────────────────────────────────
# For Colab: point RLF_CKPT_DIR at Drive to survive preemption, e.g.:
#   RLF_CKPT_DIR = Path("/content/drive/MyDrive/rlf_1_4b")
# NOTE: CKPT_EVERY=500 > smoke steps (50/100/25) so smoke only writes terminal ckpt per phase.


### 2b. Load Base Model + Tokenizer


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")
tokenizer.pad_token = tokenizer.eos_token

# Local check only — canonical HALT_ID is imported from rlf_engine_1_4b in Cell 2c (overwrites this).
HALT_ID_LOCAL = tokenizer.encode("§")[0]   # token 7803 in GPT-NeoX
print(f"HALT token '§' → ID {HALT_ID_LOCAL} (canonical HALT_ID comes from engine import next)")

# Load backbone — only if mamba_ssm is available
if mamba_ssm is not None:
    print(f"Loading base model: {BASE_MODEL}")
    # NOTE T4 sm_75 uses fp16 here; RLF bf16 params are cast to DTYPE at build (Cell 2d) for smoke stability.
    backbone = mamba_ssm.MambaLMHeadModel.from_pretrained(
        BASE_MODEL,
        dtype=DTYPE,
        device=DEVICE,
    )
    print(f"  d_model={backbone.config.d_model}, n_layers={backbone.config.n_layer}")
else:
    print("[SKIP] mamba-ssm not available — skipping model load.")
    backbone = None


### 2c. Import RLF Engine Components

The RLF engine lives in `rlf_engine_1_4b.py`. Import it to build the full
RecursiveMamba1_PrefixScratchpad model.


In [ ]:
sys.path.insert(0, str(REPO_ROOT))
from rlf_engine_1_4b import (
    RecursiveMamba1_PrefixScratchpad,
    freeze_for_phase3a,
    freeze_for_phase3b,
    freeze_for_phase3c,
    HALT_ID,
)


### 2d. Build RLF Engine


In [ ]:
if backbone is not None:
    model = RecursiveMamba1_PrefixScratchpad(backbone, lora_rank=LORA_RANK)
    model = model.to(DEVICE)
    model._print_param_report()
else:
    print("[SKIP] backbone not loaded — skipping model build.")
    model = None


### 2e. RLF Dataset


In [ ]:
from rlf_dataset import RLFDataset, collate_rlf
from torch.utils.data import DataLoader

rlf_size = SMOKE_TEST_SIZE if SMOKE_TEST else 2000
rlf_ds = RLFDataset(size=rlf_size, seq_len=64 if SMOKE_TEST else 256, adversarial_prob=0.0)
rlf_loader = DataLoader(
    rlf_ds,
    batch_size=BATCH_SIZE,
    collate_fn=collate_rlf,
    shuffle=True,
)
print(f"RLF dataset: {len(rlf_ds)} samples (smoke_test={SMOKE_TEST})")

SFT_DATA = REPO_ROOT / "combined_training_data.jsonl"
if not SFT_DATA.exists():
    print(f"[WARN] {SFT_DATA} not found — Phase 3c will use RLF data as fallback.")
    sft_ds = None
else:
    from rlf_trainer_1_4b import SFTDataset
    sft_ds = SFTDataset(str(SFT_DATA))

sft_loader = (
    DataLoader(sft_ds, batch_size=1, shuffle=True)
    if sft_ds and len(sft_ds) > 0 else None
)
print(f"SFT dataset: {len(sft_ds) if sft_ds else 0} samples")


### 2f. Training Utilities


In [ ]:
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_ckpt
import time, logging

def setup_logging(log_path: Path) -> logging.Logger:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s",
        handlers=[logging.FileHandler(log_path), logging.StreamHandler()],
    )
    return logging.getLogger(__name__)

def sft_loss(model, input_ids):
    model.train()
    B = input_ids.shape[0]
    x, res = model._encode(input_ids)
    x_prompt   = x.detach().clone()
    res_prompt = res.detach().clone() if res is not None else None

    mem     = model.latent_memory.expand(B, -1, -1)
    x_ext   = torch.cat([mem, x], dim=1)
    if res is not None:
        res_pad = torch.zeros(B, model.M, model.d_model, device=res.device, dtype=res.dtype)
        res_ext = torch.cat([res_pad, res], dim=1)
    else:
        res_ext = None

    x_ext, res_ext = model._lifeline_inject(x_ext, res_ext, x_prompt, res_prompt)
    x_ext = model.loop_rope(x_ext, 0)
    if res_ext is not None:
        res_ext = model.loop_rope(res_ext, 0)
    x_ext, res_ext = model._run_top_layers(x_ext, res_ext)
    x_ext = x_ext + model.mamba1_loop(x_ext)
    x_ext = model.loop_norm(x_ext)
    x_bridged = x_ext + model.bridge_up(model.bridge_down(x_ext))

    x_out    = x_bridged[:, model.M :, :]
    r_out    = res_ext[:, model.M :, :] if res_ext is not None else None
    x_normed = model._apply_norm(x_out, r_out)
    logits   = model.lm_head(x_normed)

    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    return F.cross_entropy(
        shift_logits.view(-1, shift_logits.shape[-1]),
        shift_labels.view(-1),
        ignore_index=tokenizer.pad_token_id,
    )

from rlf_trainer_1_4b import save_ckpt as _save_ckpt_hdd  # noqa: F401 (kept for compat; writes to /hdd_data — DO NOT USE in notebook)

def save_ckpt(model, step, phase, val_loss, ckpt_root=None):
    """Notebook-local saver: writes shards under ckpt_root (Drive-safe) + wrapped full_state.pt for exporter."""
    import torch as _t
    root = Path(ckpt_root) if ckpt_root else RLF_CKPT_DIR
    ckpt = root / f"phase{phase}_step{step:06d}"
    ckpt.mkdir(parents=True, exist_ok=True)
    _t.save(model.latent_memory.data, ckpt / "latent_memory.pt")
    _t.save(model.bridge_down.state_dict(), ckpt / "bridge_down.pt")
    _t.save(model.bridge_up.state_dict(), ckpt / "bridge_up.pt")
    _t.save(model.mamba1_loop.state_dict(), ckpt / "mamba1_loop.pt")
    _t.save(model.loop_norm.state_dict(), ckpt / "loop_norm.pt")
    _t.save(model.lifeline_gate.data, ckpt / "lifeline_gate.pt")
    _t.save(model.lm_head.state_dict(), ckpt / "lm_head.pt")
    lora_state = {n: p.data for n, p in model.named_parameters() if "lora_A" in n or "lora_B" in n}
    _t.save(lora_state, ckpt / "lora.pt")
    # Wrapped monolith for export_mamba_1_4b.export_checkpoint (expects metadata + state_dict)
    _t.save({"model_state_dict": model.state_dict(), "d_model": D_MODEL, "halt_id": HALT_ID, "prefix_m": PREFIX_M, "has_bridge": True, "base_split": BASE_SPLIT, "max_rlf_loops": MAX_LOOPS, "rope_base": 10000}, ckpt / "full_state.pt")
    log.info(f"Checkpoint: {ckpt} | val={val_loss:.4f}")

def run_phase(model, phase, log, resume_step=0):
    if model is None:
        log.warning("Model is None — cannot run phase. Skipping.")
        return
    cfg = ACTIVE_PHASE_CONFIG[phase]
    if phase == "3a":
        freeze_for_phase3a(model)
    elif phase == "3b":
        freeze_for_phase3b(model)
    else:
        freeze_for_phase3c(model)

    param_groups = []
    if cfg["lr_mem"] > 0:
        param_groups.append({"params": [model.latent_memory], "lr": cfg["lr_mem"], "weight_decay": 0.0})
    bridge_params = list(model.bridge_down.parameters()) + list(model.bridge_up.parameters())
    if cfg["lr_bridge"] > 0:
        param_groups.append({"params": bridge_params, "lr": cfg["lr_bridge"], "weight_decay": 0.01})
    other_params = [
        p for p in model.parameters()
        if p.requires_grad and p is not model.latent_memory and not any(p is bp for bp in bridge_params)
    ]
    if cfg["lr_other"] > 0 and other_params:
        param_groups.append({"params": other_params, "lr": cfg["lr_other"], "weight_decay": 0.01})

    optimizer = torch.optim.AdamW(param_groups, weight_decay=0.01)
    optimizer.zero_grad()  # clear stale grads from prior phase (trainer parity)

    if phase == "3c":
        dataloader = sft_loader if sft_loader else rlf_loader
    else:
        dataloader = rlf_loader

    if dataloader is None:
        log.warning(f"No dataloader for phase {phase}; skipping.")
        return

    data_iter = iter(dataloader)
    total_loss = 0.0
    avg_loss = 0.0
    step = resume_step

    log.info(f"Phase {phase}: {cfg['steps']} steps, batch={BATCH_SIZE}, accum={GRAD_ACCUM}")

    while step < cfg["steps"]:
        model.train()
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            batch = next(data_iter)

        t0 = time.perf_counter()
        if phase == "3c":
            # SFT fallback: rlf_loader yields (input_ids, chain, starts) tuple, SFTDataset yields Tensor
            _ids = batch[0] if isinstance(batch, (tuple, list)) else batch
            input_ids = _ids.to(DEVICE)
            loss = sft_loss(model, input_ids) / GRAD_ACCUM
        else:
            input_ids, chain_targets, ans_starts = batch
            input_ids = input_ids.to(DEVICE)
            loss, acc, ans_acc, halt_acc = model(input_ids, chain_targets, ans_starts)
            loss = loss / GRAD_ACCUM

        loss.backward()
        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            optimizer.zero_grad()

        step += 1
        dt = time.perf_counter() - t0

        if step % LOG_EVERY == 0:
            avg_loss = total_loss / LOG_EVERY
            total_loss = 0.0
            vram = torch.cuda.memory_allocated() / 1e9 if DEVICE == 'cuda' else 0.0
            log.info(
                f"[Phase{phase}][S{step:05d}] Loss={avg_loss:.4f} "
                f"| VRAM={vram:.2f}GB | {dt*1000:.0f}ms/step"
            )

        if step % CKPT_EVERY == 0 or step == cfg["steps"]:
            save_ckpt(model, step, phase, avg_loss)

    log.info(f"Phase {phase} complete at step {step}.")


### 2g. Run Phases 3a → 3b → 3c

**⚠️ Long-running cells.** Each phase can take hours on consumer GPUs.
Use `--resume` to continue from the latest checkpoint if training is interrupted.

> **Note on halting:** This notebook uses the HALT token (§) approach from the 1.4B RLF engine.
> The probability-based `HaltingHead` (Phase 14) is **not** used here.


> **Before running training:** Ensure you are on a CUDA-enabled Linux machine.
> This notebook will auto-skip training cells if CUDA is unavailable.


In [ ]:
if DEVICE != "cuda":
    print(f"[SKIP] Training requires CUDA. Current DEVICE={DEVICE}.")
    print("       Run this notebook on a CUDA machine for training.")
    print("       Export and verification cells still work on CPU/MPS.")


In [ ]:
if model is None:
    print("[SKIP] Model not built (mamba-ssm unavailable). Training cells skipped.")
else:
    LOG_PATH = REPO_ROOT / "rlf_trainer.log"
    log = setup_logging(LOG_PATH)

    phases = ["3a", "3b", "3c"]
    for i, phase in enumerate(phases):
        # Notebook-local run_phase already reads ACTIVE_PHASE_CONFIG (SMOKE vs FULL).
        # Do NOT monkey-patch rlf_trainer_1_4b.PHASE_CONFIG (dead code in prior version).
        log.info(f"\n{'='*60}\n  Starting Phase {phase}\n{'='*60}")
        run_phase(model, phase, log, resume_step=0)

    final_dir = RLF_CKPT_DIR / "final"
    final_dir.mkdir(exist_ok=True, parents=True)
    # Wrapped checkpoint for exporter (metadata required by export_mamba_1_4b.py)
    torch.save({"model_state_dict": model.state_dict(), "d_model": D_MODEL, "halt_id": HALT_ID, "prefix_m": PREFIX_M, "has_bridge": True, "base_split": BASE_SPLIT, "max_rlf_loops": MAX_LOOPS, "rope_base": 10000}, final_dir / "full_state.pt")
    torch.save(model.state_dict(), final_dir / "full_model.pt")
    log.info(f"Final checkpoint saved to: {final_dir}")
    print("Training complete. If on Colab, copy to Drive: !cp -r '%s' /content/drive/MyDrive/  # adjust" % str(final_dir))


## 3. Fuse LoRA + Save Final Checkpoint

Before export, merge LoRA A/B into the base weights so the exported model
has plain linear layers with no runtime LoRA overhead.


In [ ]:
from rlf_engine_1_4b import fuse_lora_weights

if model is None:
    print("[SKIP] model is None (CPU-only path) — skipping LoRA fuse. Train on CUDA first.")
else:
    print("Fusing LoRA weights into base layers...")
    fuse_lora_weights(model)
    print("LoRA fusion complete.")

    final_dir = RLF_CKPT_DIR / "final"
    final_dir.mkdir(exist_ok=True, parents=True)
    # Re-save wrapped + raw after fuse (fused weights are plain Linear)
    try:
        torch.save({"model_state_dict": model.state_dict(), "d_model": D_MODEL, "halt_id": HALT_ID, "prefix_m": PREFIX_M, "has_bridge": True, "base_split": BASE_SPLIT, "max_rlf_loops": MAX_LOOPS, "rope_base": 10000}, final_dir / "full_state.pt")
        torch.save(model.state_dict(), final_dir / "full_model.pt")
        print(f"Final checkpoint saved to: {final_dir}")
    except NameError as e:
        print(f"[WARN] D_MODEL/HALT_ID not in scope (run Config + Engine cells first): {e}")


## 4. Export to `.mamba.bin` (Mobile Format)

The project's `export_mamba_1_4b.py` serializes the full RLF model
(backbone + loop engine + scratchpad + bridge) into a flat binary format
that the mobile C engine (`ssm_infer.c`) loads in milliseconds.

> **Note:** The Expo app looks for `mamba_latent_Q4_K_M.mamba.bin` but the export
> produces `mamba_latent.mamba.bin`. Rename or update the expected filename
> in `apps/expo/App.tsx`.


In [ ]:
import sys as _sys
if not (REPO_ROOT / "export_mamba_1_4b.py").exists():
    raise FileNotFoundError(
        "export_mamba_1_4b.py is UNTRACKED in git origin and missing after clone. "
        "Push it from your Mac (git add export_mamba_1_4b.py) or upload via Colab Files to REPO_ROOT, then re-run."
    )
from export_mamba_1_4b import export_checkpoint as export_m1, QUANT_INT8, QUANT_FP32

EXPORT_DIR = REPO_ROOT / "mobile" / "model"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
# Trainer/notebook wrapped ckpt preferred; fall back to raw state_dict
FULL_CKPT_WRAPPED = RLF_CKPT_DIR / "final" / "full_state.pt"
FULL_CKPT_RAW = RLF_CKPT_DIR / "final" / "full_model.pt"
FULL_CKPT = FULL_CKPT_WRAPPED if FULL_CKPT_WRAPPED.exists() else FULL_CKPT_RAW
OUTPUT_BIN = EXPORT_DIR / "mamba_latent.mamba.bin"
# Expo App.tsx expects mamba_latent_Q4_K_M.mamba.bin — copy after export (no re-encode)
OUTPUT_BIN_Q4 = EXPORT_DIR / "mamba_latent_Q4_K_M.mamba.bin"

if FULL_CKPT.exists():
    # NOTE: C loader (mobile/cpp/ssm_weights.c) is fp32-only today — int8 layout has no dequant.
    # Use FP32 for first mobile bring-up; switch to QUANT_INT8 only after loader gains dequant.
    export_m1(
        ckpt_path=str(FULL_CKPT),
        output_path=str(OUTPUT_BIN),
        quant_type=QUANT_FP32,
    )
    print(f"\nExported model: {OUTPUT_BIN}")
    print(f"Size: {OUTPUT_BIN.stat().st_size / 1e6:.1f} MB")
    import shutil
    shutil.copyfile(OUTPUT_BIN, OUTPUT_BIN_Q4)
    print(f"Copied for Expo (App.tsx MODEL_ASSET_NAME): {OUTPUT_BIN_Q4}")
else:
    print("[WARN] No checkpoint found at", FULL_CKPT_WRAPPED, "nor", FULL_CKPT_RAW)
    print("       Run training cells first (expect checkpoints/rlf_1_4b/final/full_state.pt).")


## 5. Export BPE Tokenizer (`.bpe.bin`)

The mobile C engine includes a lightweight BPE tokenizer (`bpe_tokenizer.c`).
Export the GPT-NeoX vocabulary + merge table into the binary format it expects.


In [ ]:
from export_bpe_table import export_bpe

BPE_OUT = EXPORT_DIR / "tokenizer.bpe.bin"
export_bpe(str(BPE_OUT))
print(f"\nExported tokenizer: {BPE_OUT}")
print(f"Size: {BPE_OUT.stat().st_size / 1024:.1f} KB")

# WARNING vocab drift: export_bpe_table adds <THINK>/<HALT> (+2) vs training vocab (GPT-NeoX + §=7803).
# For smoke bring-up this is OK (tokenizer only used by C engine), but header vocab_size (from embedding)
# will be base, .bpe.bin vocab will be base+2. Keep HALT as § for training; <HALT> is a separate id.


## 6. Verify Exported Artifacts


In [ ]:
def verify_mamba_bin(path: Path) -> None:
    import struct
    data = path.read_bytes()
    magic = struct.unpack_from("<I", data, 0)[0]
    assert magic == 0x4D414D42, f"Bad magic: {hex(magic)}"
    version = struct.unpack_from("<I", data, 4)[0]
    assert version == 2, f"Bad version: {version}"
    # Header v2: <II25iQ — d_model@8, d_state@12, d_conv@16, expand@20, n_layers@24, vocab@28, total_bytes@108
    d_model = struct.unpack_from("<i", data, 8)[0]
    n_layers = struct.unpack_from("<i", data, 24)[0]
    vocab_size = struct.unpack_from("<i", data, 28)[0]
    total_bytes = struct.unpack_from("<Q", data, 108)[0]
    print(f"  ✅ {path.name}")
    print(f"     d_model={d_model}, n_layers={n_layers}, vocab={vocab_size}")
    print(f"     total_bytes={total_bytes:,}, actual={len(data):,}, match={'✅' if total_bytes==len(data) else '❌'}")
    # NOTE: C struct (ssm_weights.h) is 120B (26x int32) vs exporter 116B — C loader misparses until fixed.

def verify_bpe_bin(path: Path) -> None:
    import struct
    data = path.read_bytes()
    magic = data[:4]
    assert magic == b"BPE\x00", f"Bad magic: {magic}"
    vocab_size, merge_count, max_len = struct.unpack_from("<III", data, 4)
    print(f"  ✅ {path.name}")
    print(f"     vocab={vocab_size}, merges={merge_count}, max_token_len={max_len}")

print("Verifying exported artifacts:\n")
for _p, _fn in [(OUTPUT_BIN, verify_mamba_bin), (BPE_OUT, verify_bpe_bin)]:
    if not _p.exists():
        print(f"  ⏭️  SKIP {_p.name}: not found at {_p} — run export cells first.")
    else:
        _fn(_p)


## 7. Next Steps

1. **Copy artifacts to mobile app bundle:**
   - iOS: Add `mobile/model/*.mamba.bin` and `tokenizer.bpe.bin` to Xcode project → Build Phases → Copy Bundle Resources
   - Android: Place in `android/app/src/main/assets/`

2. **Build the Expo development client:**
   ```bash
   cd apps/expo
   npm install
   npx expo run:ios    # or: npx expo run:android
   ```

3. **Load and run inference from the app:**
   ```typescript
   import MambaNative from 'mamba-ssm';

   // iOS: NSBundle.mainBundle.path(forResource: "mamba_latent_Q4_K_M", ofType: "mamba.bin")
   // Android: filesDir path or asset path
   await MambaNative.loadModel(modelPath);
   const result = await MambaNative.runInference(tokenIds);
   console.log(`Loops: ${result.nLoops}, finalToken: ${result.finalToken}`);
   ```
